# Distances Between Observations

Read this notebook from top to bottom and fill in the code as you go. Work together and discuss with other students in the class. Try to resolve any errors on your own first, but don't get stuck; ask for help!

In addition to writing and running code, be sure to examine any output and interpret the results before moving on.

For many of these questions, there are several approaches, and there is no single right answer. You should try a few different things and compare with your classmates.

We will use `scikit-learn` extensively later, but for this activity you might want to stick with `pandas`.


In [9]:
import pandas as pd
import numpy as np

## Ames - Recommending Similar Homes

1\. Suppose that you really like house 0 in the Ames housing data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar (based on these 3 variables). Do they make sense?

_Think:_ If the goal is to find a "good deal" on a similar house, should sale price be included as a variable in your distance metric?

In [10]:
df_ames = pd.read_csv("https://raw.githubusercontent.com/kevindavisross/data301/main/data/AmesHousing.txt", sep="\t")
df_ames["Bathrooms"] = df_ames["Full Bath"] + 0.5 * (df_ames["Half Bath"])
df_ames[["Bedroom AbvGr", "Gr Liv Area", "Bathrooms", "SalePrice", "House Style", "Neighborhood", "Year Built"]]

,Bedroom AbvGr,Gr Liv Area,Bathrooms,SalePrice,House Style,Neighborhood,Year Built
0,3,1656,1.0,215000,1Story,NAmes,1960
1,2,896,1.0,105000,1Story,NAmes,1961
2,3,1329,1.5,172000,1Story,NAmes,1958
3,3,2110,2.5,244000,1Story,NAmes,1968
4,3,1629,2.5,189900,2Story,Gilbert,1997
...,...,...,...,...,...,...,...
2925,3,1003,1.0,142500,SLvl,Mitchel,1984
2926,2,902,1.0,131000,1Story,Mitchel,1983
2927,3,970,1.0,132000,SFoyer,Mitchel,1992
2928,2,1389,1.0,170000,1Story,Mitchel,1974


In [11]:
house0 = df_ames.loc[0]
show_vars = ["Bedroom AbvGr", "Gr Liv Area", "Bathrooms", "SalePrice", "House Style", "Neighborhood", "Year Built"]
house0[show_vars]

Bedroom AbvGr         3
Gr Liv Area        1656
Bathrooms           1.0
SalePrice        215000
House Style      1Story
Neighborhood      NAmes
Year Built         1960
Name: 0, dtype: object

In [30]:
features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms"]
X = df_ames[features].astype(float)

X_z = (X - X.mean()) / X.std()



diff = X_z - X_z.loc[0]


df_ames["dist_euclid"] = np.sqrt((diff ** 2).sum(axis=1))


cheaper = df_ames[df_ames["SalePrice"] < house0["SalePrice"]]
show = show_vars + ["dist_euclid"]

cheaper.sort_values("dist_euclid")[show].head(20)

,Bedroom AbvGr,Gr Liv Area,Bathrooms,SalePrice,House Style,Neighborhood,Year Built,dist_euclid
1226,3,1661,1.0,165500,SLvl,NAmes,1955,0.009891
1940,3,1647,1.0,153000,1Story,NAmes,1953,0.017804
1357,3,1666,1.0,161000,2Story,OldTown,1925,0.019782
758,3,1666,1.0,135000,1.5Fin,IDOTRR,1927,0.019782
291,3,1666,1.0,100000,1.5Fin,SWISU,1931,0.019782
2637,3,1668,1.0,135000,1.5Fin,OldTown,1948,0.023738
618,3,1644,1.0,167000,1Story,NAmes,1953,0.023738
2700,3,1640,1.0,131000,1Story,Sawyer,1950,0.031651
1529,3,1639,1.0,115000,1.5Fin,SWISU,1936,0.033629
179,3,1633,1.0,129000,1.5Fin,OldTown,1948,0.045499


I excluded sale price from the distance and used it only to filter for cheaper homes. The closest homes have 3 bedrooms, 1 bathroom, and about 1,650 square feet, so they are sensible matches for house 0.

2\. Continuing part 1. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms, **and House Style** --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

In [ ]:
style = pd.get_dummies(df_ames[["House Style"]], dtype=float)
X_style = pd.concat([X_z, style], axis=1)


diff_style = X_style - X_style.loc[0]

# Euclidean distance
df_ames["dist_style_euclid"] = np.sqrt((diff_style ** 2).sum(axis=1))


cheaper = df_ames[df_ames["SalePrice"] < house0["SalePrice"]]
show_style = show_vars + ["dist_style_euclid"]

cheaper.sort_values("dist_style_euclid")[show_style].head(5)

,Bedroom AbvGr,Gr Liv Area,Bathrooms,SalePrice,House Style,Neighborhood,Year Built,dist_style_euclid
1940,3,1647,1.0,153000,1Story,NAmes,1953,0.018
618,3,1644,1.0,167000,1Story,NAmes,1953,0.024
2700,3,1640,1.0,131000,1Story,Sawyer,1950,0.032
314,3,1687,1.0,160000,1Story,Timber,1948,0.061
788,3,1689,1.0,127500,1Story,Edwards,1956,0.065


Yes, they make sense

3\. Continuing parts 1 and 2. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it, by calculating distances. You can **choose the variables to include, but include both quantitative and categorical variables**. Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

_Hint:_ There are many variables in the data set. Do not attempt to compute distance based on all the variables! You will want to pare down the number of variables, but be sure to include a mixture of categorical and quantitative variables. Refer to the [data documentation](https://ww2.amstat.org/publications/jse/v19n3/decock/DataDocumentation.txt) for information about the variables.


In [ ]:
quant_features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "Year Built"]
categorical_features = ["House Style", "Neighborhood"]

X_quant = df_ames[quant_features].astype(float)
X_quant_z = (X_quant - X_quant.mean()) / X_quant.std()

X_categorical = pd.get_dummies(df_ames[categorical_features], dtype=float)
X_selected = pd.concat([X_quant_z, X_categorical], axis=1)

diff_selected = X_selected - X_selected.loc[0]

# Euclidean distance
df_ames["dist_selected"] = np.sqrt((diff_selected ** 2).sum(axis=1))

cheaper = df_ames[df_ames["SalePrice"] < house0["SalePrice"]]
show_selected = show_vars + ["dist_selected"]

cheaper.sort_values("dist_selected")[show_selected].head(5)

,Bedroom AbvGr,Gr Liv Area,Bathrooms,SalePrice,House Style,Neighborhood,Year Built,dist_selected
1240,3,1570,1.0,166800,1Story,NAmes,1958,0.183
1940,3,1647,1.0,153000,1Story,NAmes,1953,0.232
618,3,1644,1.0,167000,1Story,NAmes,1953,0.233
2558,3,1433,1.0,161000,1Story,NAmes,1961,0.442
1896,3,1429,1.0,181900,1Story,NAmes,1960,0.449


I used standardized Euclidean distance with living area, bedrooms, bathrooms, year built, house style, and neighborhood. The closest matches were similar one-story homes in NAmes, with house 1240 ranked first.

## Colleges similar to Cal Poly

We'll use data from the [College Scorecard data](https://collegescorecard.ed.gov/) to find colleges and universities that are similar to Cal Poly.

In [35]:
df_college = pd.read_csv("https://datasci112.stanford.edu/data/college_attributes.csv")

df_college.set_index("Institution", inplace = True)

df_college

,City,State,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,PCIP01,PCIP03,PCIP04,PCIP05,...,PCIP44,PCIP45,PCIP46,PCIP47,PCIP48,PCIP49,PCIP50,PCIP51,PCIP52,PCIP54
Institution,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,Normal,AL,0.7160,5098.0,Master's Colleges & Universities: Larger Programs,Public,0.0445,0.0071,0.0053,0.0000,...,0.0409,0.0249,0.0,0.0,0.0,0.0,0.0231,0.0000,0.1637,0.0000
University of Alabama at Birmingham,Birmingham,AL,0.8854,13284.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0020,...,0.0195,0.0239,0.0,0.0,0.0,0.0,0.0249,0.2088,0.2159,0.0141
University of Alabama in Huntsville,Huntsville,AL,0.7367,7358.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0127,0.0,0.0,0.0,0.0,0.0407,0.1341,0.1930,0.0073
Alabama State University,Montgomery,AL,0.9799,3495.0,Doctoral/Professional Universities,Public,0.0000,0.0000,0.0000,0.0000,...,0.0648,0.0196,0.0,0.0,0.0,0.0,0.0511,0.0904,0.1513,0.0059
The University of Alabama,Tuscaloosa,AL,0.7890,30725.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0061,0.0000,0.0019,...,0.0072,0.0661,0.0,0.0,0.0,0.0,0.0234,0.1077,0.2916,0.0096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Florida Academy of Nursing,Miramar,FL,0.3088,239.0,Not applicable,Private for-profit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,1.0000,0.0000,0.0000
Herzing University-Tampa,Tampa,FL,0.9630,68.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000
Abilene Christian University-Undergraduate Online,Addison,TX,1.0000,415.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000


We'll want to single out Cal Poly, which we can do like this.

In [38]:
school_name = "California Polytechnic State University-San Luis Obispo"

cp = df_college.loc[school_name]

cp

City                                                        San Luis Obispo
State                                                                    CA
AdmissionRate                                                          0.33
Undergraduates                                                      21090.0
CarnegieClassification    Master's Colleges & Universities: Larger Programs
Ownership                                                            Public
PCIP01                                                               0.1084
PCIP03                                                               0.0255
PCIP04                                                               0.0441
PCIP05                                                               0.0019
PCIP09                                                               0.0353
PCIP10                                                               0.0175
PCIP11                                                               0.0326
PCIP12      

1\. Based on only the admission rate and the number of undergraduates, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [46]:
college_features = ["AdmissionRate", "Undergraduates"]


college_complete = df_college.dropna(subset=college_features).copy()
X_college = college_complete[college_features].astype(float)
X_college_z = (X_college - X_college.mean()) / X_college.std()

diff_college = X_college_z - X_college_z.loc[school_name]

college_complete["dist_euclid"] = np.sqrt(
    (diff_college ** 2).sum(axis=1)
)
college_show = ["City", "State", "AdmissionRate", "Undergraduates", "dist_euclid"]

college_complete.drop(index=school_name).sort_values("dist_euclid")[
    college_show
].head(7)

,City,State,AdmissionRate,Undergraduates,dist_euclid
Institution,,,,,
University of California-Santa Barbara,Santa Barbara,CA,0.2918,23081.0,0.309162
DeVry University-Illinois,Naperville,IL,0.4552,19729.0,0.593121
University of North Carolina at Chapel Hill,Chapel Hill,NC,0.2040,19722.0,0.596846
Clemson University,Clemson,SC,0.4922,21577.0,0.736788
University of Virginia-Main Campus,Charlottesville,VA,0.2074,17041.0,0.761296
CUNY Hunter College,New York,NY,0.4590,17293.0,0.761441
Stony Brook University,Stony Brook,NY,0.4806,17900.0,0.795756


After standardizing admission rate and undergraduate enrollment, it showed that UC Santa Barbara, DeVry University, and UNC, and more were similar to Cal Poly

2\. Now consider the admission rate, the number of undergraduates, and also the [Carnegie classification](https://en.wikipedia.org/wiki/Carnegie_Classification_of_Institutions_of_Higher_Education) of the type of school, and the ownership (public, private, etc.) Based on these variables, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [48]:
college_quantitative = ["AdmissionRate", "Undergraduates"]
college_categorical = ["CarnegieClassification", "Ownership"]

college_all = df_college.dropna(
    subset=college_quantitative + college_categorical
).copy()
X_college_quant = college_all[college_quantitative].astype(float)
X_college_quant_z = (
    (X_college_quant - X_college_quant.mean()) / X_college_quant.std()
)



X_college_categorical = pd.get_dummies(
    college_all[college_categorical], dtype=float
)
X_college_all = pd.concat(
    [X_college_quant_z, X_college_categorical], axis=1
)


college_all["dist_all_euclid"] = np.sqrt(
    ((X_college_all - X_college_all.loc[school_name]) ** 2).sum(axis=1)
)

college_all_show = [
    "City", "State", "AdmissionRate", "Undergraduates",
    "CarnegieClassification", "Ownership", "dist_all_euclid",
]

display(
    college_all.drop(index=school_name)
    .sort_values("dist_all_euclid")[college_all_show]
    .head(10)
)


,City,State,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,dist_all_euclid
Institution,,,,,,,
CUNY Hunter College,New York,NY,0.4590,17293.0,Master's Colleges & Universities: Larger Programs,Public,0.761441
CUNY Bernard M Baruch College,New York,NY,0.5056,15483.0,Master's Colleges & Universities: Larger Programs,Public,1.073601
CUNY John Jay College of Criminal Justice,New York,NY,0.4458,12834.0,Master's Colleges & Universities: Larger Programs,Public,1.184989
CUNY Brooklyn College,Brooklyn,NY,0.5136,12567.0,Master's Colleges & Universities: Larger Programs,Public,1.376321
University of California-Santa Barbara,Santa Barbara,CA,0.2918,23081.0,Doctoral Universities: Very High Research Acti...,Public,1.447612
California State Polytechnic University-Pomona,Pomona,CA,0.6062,26802.0,Master's Colleges & Universities: Larger Programs,Public,1.450297
CUNY Queens College,Queens,NY,0.6078,14859.0,Master's Colleges & Universities: Larger Programs,Public,1.491386
DeVry University-Illinois,Naperville,IL,0.4552,19729.0,Master's Colleges & Universities: Larger Programs,Private for-profit,1.533555
University of North Carolina at Chapel Hill,Chapel Hill,NC,0.2040,19722.0,Doctoral Universities: Very High Research Acti...,Public,1.535000


I one-hot encoded Carnegie classification and ownership before calculating Euclidean distance and these were the results ^

3\. The columns whose names begin with "PCIP" contain the proportions of students at each school studying various fields (e.g., Engineering, Psychology). Each field is represented by a two-digit code called the [CIP code](https://nces.ed.gov/ipeds/cipcode/browse.aspx?y=55).

If we only consider the proportions of students studying various fields, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [50]:
# PCIP rows are profiles, so compare their direction with cosine similarity.
pcip_columns = [column for column in df_college.columns if column.startswith("PCIP")]
X_pcip = df_college[pcip_columns].astype(float)
cp_profile = X_pcip.loc[school_name]
school_norms = np.sqrt((X_pcip ** 2).sum(axis=1))
cp_norm = np.sqrt((cp_profile ** 2).sum())
df_college["cosine_similarity"] = (
    (X_pcip @ cp_profile) / (school_norms * cp_norm)
)

display(
    df_college.drop(index=school_name)
    .sort_values("cosine_similarity", ascending=False)
    [["City", "State", "cosine_similarity"]]
    .head(5)
    .round(5)
)


,City,State,cosine_similarity
Institution,,,
North Carolina State University at Raleigh,Raleigh,NC,0.96857
Iowa State University,Ames,IA,0.96725
University of Illinois Urbana-Champaign,Champaign,IL,0.94126
Mississippi State University,Mississippi State,MS,0.93275
Texas A & M University-College Station,College Station,TX,0.92681


I used cosine similarity because the PCIP columns describe each school's mix of fields. NC State, Iowa State, UIUC, Mississippi State, and Texas A&M were most similar to Cal Poly.